# Sample Questions — Apprentissage supervisé (correction détaillée)

Correction des **10 problèmes** du sujet `Sample_Questions.pdf`. Pour chaque question :
1. le **raisonnement** et les **calculs à la main** (ce qui est attendu à l'examen) ;
2. une courte **vérification en Python** (et parfois un graphique) pour contrôler les résultats.

| # | Thème | Notions clés |
|---|---|---|
| 1 | k plus proches voisins | distance euclidienne, vote majoritaire, mise à l'échelle |
| 2 | Bayes naïf | indépendance conditionnelle, formule de Bayes, probabilité a posteriori |
| 3 | Arbre de classification | impureté de Gini, diminution d'impureté |
| 4 | Classification à coûts | coût espéré, seuil optimal |
| 5 | Arbre de régression | moyenne des feuilles, erreur quadratique |
| 6 | Matrice de confusion | accuracy, précision, rappel, FPR, F1, classes déséquilibrées |
| 7 | Courbe ROC | TPR / FPR, AUC par comptage de paires, invariance |
| 8 | Régression logistique | frontière de décision, effet du seuil |
| 9 | Perte logistique | erreur de classification vs log-loss |
| 10 | Validation croisée | choix d'hyperparamètre, rôle du jeu de test |

In [1]:
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import log_loss, roc_auc_score

sys.path += [".", ".."]          # plot_theme.py est a la racine du projet
from plot_theme import *         # couleurs, pick_color(), theme sombre applique par defaut

---
## Problème 1 — k plus proches voisins et mise à l'échelle

| Observation | $X_1$ | $X_2$ | $Y$ |
|---|---|---|---|
| A | 1 | 8 | 0 |
| B | 3 | 3 | 1 |
| C | 5 | 5 | 1 |
| D | 7 | 2 | 0 |
| E | 8 | 7 | 0 |

On veut classer le nouveau point $x = (4, 4)$.

**Rappel.** Le $k$-NN prédit la **classe majoritaire** parmi les $k$ observations d'apprentissage les plus
proches de $x$. Avec la distance euclidienne :
$$d(x, x_i) = \sqrt{(x_1 - x_{i1})^2 + (x_2 - x_{i2})^2}.$$

### Question 1 — Distances euclidiennes

| Obs. | $(x_1 - x_{i1},\ x_2 - x_{i2})$ | $d^2$ | $d$ |
|---|---|---|---|
| A | $(3,\ -4)$ | $9 + 16 = 25$ | $5$ |
| B | $(1,\ 1)$ | $1 + 1 = 2$ | $\sqrt{2} \approx 1.414$ |
| C | $(-1,\ -1)$ | $1 + 1 = 2$ | $\sqrt{2} \approx 1.414$ |
| D | $(-3,\ 2)$ | $9 + 4 = 13$ | $\sqrt{13} \approx 3.606$ |
| E | $(-4,\ -3)$ | $16 + 9 = 25$ | $5$ |

Classement du plus proche au plus lointain : **B = C** ($1.414$) < **D** ($3.606$) < **A = E** ($5$).

### Question 2 — Prédictions 1-NN et 3-NN

- **1-NN** : le plus proche voisin est B **ou** C (égalité à $\sqrt 2$). Les deux sont de classe 1, l'égalité
  ne pose donc aucun problème : $\hat y = 1$.
- **3-NN** : les trois plus proches sont B, C, D, de classes $(1, 1, 0)$ → vote majoritaire $2$ contre $1$ :
  $\hat y = 1$.

In [2]:
train = pd.DataFrame({"X1": [1, 3, 5, 7, 8], "X2": [8, 3, 5, 2, 7], "Y": [0, 1, 1, 0, 0]},
                     index=list("ABCDE"))


def distances(train, x):
    return np.sqrt(((train[["X1", "X2"]] - x) ** 2).sum(axis=1))


def predire(train, x, k):
    voisins = distances(train, x).sort_values(kind="stable").index[:k]
    return train.loc[voisins, "Y"].mode()[0]          # classe majoritaire parmi les k voisins


x = np.array([4, 4])
print(distances(train, x).round(3).to_string())
print("Prediction 1-NN :", predire(train, x, 1), "| 3-NN :", predire(train, x, 3))

A    5.000
B    1.414
C    1.414
D    3.606
E    5.000
Prediction 1-NN : 1 | 3-NN : 1


In [3]:
fig = go.Figure()
for classe in (0, 1):
    pts = train[train.Y == classe]
    fig.add_trace(go.Scatter(x=pts.X1, y=pts.X2, mode="markers+text", text=pts.index, textposition="top center",
                             marker=dict(size=14, color=CLASS_COLORS[classe]), name=f"Classe {classe}"))
fig.add_trace(go.Scatter(x=[4], y=[4], mode="markers+text", text=["x"], textposition="bottom center",
                         marker=dict(size=16, symbol="star", color=HIGHLIGHT), name="Nouveau point"))
# Cercle passant par le 3e voisin (D) : il contient les 3 plus proches voisins
r = np.sqrt(13)
fig.add_shape(type="circle", x0=4 - r, y0=4 - r, x1=4 + r, y1=4 + r, line=dict(color=MUTED, dash="dash"))
fig.update_layout(title="Probleme 1 : le voisinage 3-NN de x = (4, 4)", xaxis_title="X1", yaxis_title="X2",
                  yaxis=dict(scaleanchor="x"), height=550)
fig.show()

### Question 3 — Après la transformation $X_1' = 10\,X_1$

On multiplie $X_1$ par 10 **dans l'apprentissage et pour le point test** : $x' = (40, 4)$.

| Obs. | $X_1'$ | $(x_1' - x_{i1}',\ x_2 - x_{i2})$ | $d^2$ | $d$ |
|---|---|---|---|---|
| A | 10 | $(30,\ -4)$ | $900 + 16 = 916$ | $\approx 30.27$ |
| B | 30 | $(10,\ 1)$ | $100 + 1 = 101$ | $\approx 10.05$ |
| C | 50 | $(-10,\ -1)$ | $100 + 1 = 101$ | $\approx 10.05$ |
| D | 70 | $(-30,\ 2)$ | $900 + 4 = 904$ | $\approx 30.07$ |
| E | 80 | $(-40,\ -3)$ | $1600 + 9 = 1609$ | $\approx 40.11$ |

- **1-NN** : B ou C (égalité), tous deux de classe 1 → $\hat y = 1$.
- **3-NN** : B, C, D → $(1, 1, 0)$ → $\hat y = 1$.

**Les prédictions ne changent pas ici, mais les distances, elles, ont complètement changé de nature** :
- $X_1$ représente désormais plus de **98 %** de chaque distance au carré (ex. pour D : $900$ sur $904$) ;
- $X_2$ ne sert plus qu'à **départager** des points qui ont le même écart en $X_1$ : D passe devant A
  uniquement parce que $|4 - 2| < |4 - 8|$, alors que leurs écarts en $X_1$ sont identiques ($\pm 30$).

Le résultat est donc inchangé « par chance » : les voisins sont presque déterminés par $X_1$ seul.

In [4]:
train_10 = train.assign(X1=10 * train.X1)
x_10 = np.array([40, 4])
print(distances(train_10, x_10).round(2).to_string())
print("Prediction 1-NN :", predire(train_10, x_10, 1), "| 3-NN :", predire(train_10, x_10, 3))

A    30.27
B    10.05
C    10.05
D    30.07
E    40.11
Prediction 1-NN : 1 | 3-NN : 1


### Question 4 — Pourquoi la mise à l'échelle influence le $k$-NN

Le $k$-NN repose **entièrement** sur une distance. La distance euclidienne additionne les écarts au carré de
chaque variable : une variable mesurée sur une **grande échelle** (ou avec une grande variance) **domine** la
somme, et les autres variables deviennent quasiment invisibles. Changer l'unité d'une variable (mètres →
centimètres, ×10…) change donc **qui sont les voisins**, et potentiellement la prédiction, alors que
l'information contenue dans les données est exactement la même.

Exemple où la prédiction change : le point $x = (5, 2)$.
- échelle d'origine : plus proche voisin **D** $= (7, 2)$ à distance $2$ (C est à distance $3$) → $\hat y = 0$ ;
- avec $X_1' = 10 X_1$, $x' = (50, 2)$ : **C** $= (50, 5)$ est à distance $3$, D $= (70, 2)$ à distance $20$
  → $\hat y = 1$.

**En pratique** : on **standardise** les variables (centrer-réduire) avant un $k$-NN, avec une moyenne et un
écart-type estimés **sur l'apprentissage uniquement** puis appliqués au test.

In [5]:
x_test = np.array([5, 2])
print("Echelle d'origine : 1-NN ->", predire(train, x_test, 1))
print("X1 multiplie par 10 : 1-NN ->", predire(train_10, x_test * [10, 1], 1))

Echelle d'origine : 1-NN -> 0
X1 multiplie par 10 : 1-NN -> 1


---
## Problème 2 — Bayes naïf et probabilités a posteriori

| Classe | A priori | $P(X_1 = 1 \mid Y)$ | $P(X_2 = 1 \mid Y)$ | $P(X_3 = 1 \mid Y)$ |
|---|---|---|---|---|
| $Y = 1$ | 0.3 | 0.8 | 0.7 | 0.6 |
| $Y = 0$ | 0.7 | 0.2 | 0.4 | 0.3 |

Patient : $X_1 = 1,\ X_2 = 0,\ X_3 = 1$.

**Hypothèse naïve** : les symptômes sont **indépendants conditionnellement à la classe**, donc
$$P(X_1, X_2, X_3 \mid Y) = P(X_1 \mid Y)\,P(X_2 \mid Y)\,P(X_3 \mid Y).$$
Attention : le tableau donne $P(X_2 = 1 \mid Y)$ ; il faut utiliser $P(X_2 = 0 \mid Y) = 1 - P(X_2 = 1 \mid Y)$.

### Question 1 — Vraisemblance sachant $Y = 1$
$$P(1, 0, 1 \mid Y = 1) = 0.8 \times (1 - 0.7) \times 0.6 = 0.8 \times 0.3 \times 0.6 = \mathbf{0.144}.$$

### Question 2 — Vraisemblance sachant $Y = 0$
$$P(1, 0, 1 \mid Y = 0) = 0.2 \times (1 - 0.4) \times 0.3 = 0.2 \times 0.6 \times 0.3 = \mathbf{0.036}.$$

### Question 3 — Probabilité a posteriori (formule de Bayes)
$$P(Y = 1 \mid x) = \frac{P(x \mid Y = 1)\,P(Y = 1)}{P(x \mid Y = 1)\,P(Y = 1) + P(x \mid Y = 0)\,P(Y = 0)}.$$
- numérateur : $0.144 \times 0.3 = 0.0432$ ;
- second terme du dénominateur : $0.036 \times 0.7 = 0.0252$ ;
- dénominateur $P(x) = 0.0432 + 0.0252 = 0.0684$.

$$P(Y = 1 \mid x) = \frac{0.0432}{0.0684} = \frac{12}{19} \approx \mathbf{0.632}, \qquad
P(Y = 0 \mid x) \approx 0.368.$$

### Question 4 — Classe prédite
$P(Y = 1 \mid x) \approx 0.632 > 0.5$ : Bayes naïf prédit **$Y = 1$** (le patient a la condition).
On remarque que l'a priori était défavorable ($0.3$), mais la vraisemblance est 4 fois plus forte pour
$Y = 1$ ($0.144$ contre $0.036$), ce qui renverse la décision.

In [6]:
prior = {1: 0.3, 0: 0.7}
p_symptome = {1: [0.8, 0.7, 0.6], 0: [0.2, 0.4, 0.3]}   # P(Xj = 1 | Y)
patient = [1, 0, 1]

vraisemblance = {c: np.prod([p if xj == 1 else 1 - p for p, xj in zip(p_symptome[c], patient)]) for c in (1, 0)}
jointe = {c: prior[c] * vraisemblance[c] for c in (1, 0)}
evidence = jointe[1] + jointe[0]

print("P(x | Y=1) =", round(vraisemblance[1], 4), "| P(x | Y=0) =", round(vraisemblance[0], 4))
print("P(x) =", round(evidence, 4))
print("P(Y=1 | x) =", round(jointe[1] / evidence, 4), "-> classe predite :", int(jointe[1] > jointe[0]))

P(x | Y=1) = 0.144 | P(x | Y=0) = 0.036
P(x) = 0.0684
P(Y=1 | x) = 0.6316 -> classe predite : 1


### Question 5 — Pourquoi peut-on ignorer le dénominateur… mais pas toujours

Le dénominateur $P(x) = \sum_c P(x \mid Y = c)\,P(Y = c)$ **ne dépend pas de la classe** : c'est la même
constante positive pour $Y = 0$ et $Y = 1$.
- Pour **choisir la classe**, on compare $P(Y = 1 \mid x)$ et $P(Y = 0 \mid x)$ : diviser les deux par la même
  constante ne change pas lequel est le plus grand. Il suffit de comparer les numérateurs
  $0.0432 > 0.0252$ :
  $$\hat y = \arg\max_c P(x \mid Y = c)\,P(Y = c).$$
- Pour obtenir une **probabilité**, il faut des nombres qui **somment à 1** : $0.0432$ et $0.0252$ ne sont pas
  des probabilités a posteriori (leur somme vaut $0.0684$). Le dénominateur est justement la constante de
  **normalisation**. On en a besoin dès qu'on veut un score interprétable : seuil différent de $1/2$, coûts
  asymétriques, courbe ROC, communication d'un risque au patient…

---
## Problème 3 — Arbre de décision et impureté de Gini

Nœud parent : 20 observations, **8** de classe 0, **7** de classe 1, **5** de classe 2.

| Split | Enfant | Classe 0 | Classe 1 | Classe 2 | Total |
|---|---|---|---|---|---|
| A | Gauche | 6 | 2 | 2 | 10 |
| A | Droite | 2 | 5 | 3 | 10 |
| B | Gauche | 5 | 5 | 0 | 10 |
| B | Droite | 3 | 2 | 5 | 10 |

$$G(t) = 1 - \sum_{j=0}^{2} p_j(t)^2.$$
$G = 0$ pour un nœud **pur** ; $G$ est maximal quand les classes sont équiréparties ($1 - 1/3 = 0.667$ pour
3 classes).

### Question 1 — Gini du parent
Proportions : $(8, 7, 5)/20 = (0.40,\ 0.35,\ 0.25)$.
$$G = 1 - (0.40^2 + 0.35^2 + 0.25^2) = 1 - (0.16 + 0.1225 + 0.0625) = 1 - 0.345 = \mathbf{0.655}.$$

### Question 2 — Gini de chaque enfant
| Nœud | Proportions | $\sum p_j^2$ | $G$ |
|---|---|---|---|
| A gauche | $(0.6,\ 0.2,\ 0.2)$ | $0.36 + 0.04 + 0.04 = 0.44$ | $\mathbf{0.56}$ |
| A droite | $(0.2,\ 0.5,\ 0.3)$ | $0.04 + 0.25 + 0.09 = 0.38$ | $\mathbf{0.62}$ |
| B gauche | $(0.5,\ 0.5,\ 0)$ | $0.25 + 0.25 + 0 = 0.50$ | $\mathbf{0.50}$ |
| B droite | $(0.3,\ 0.2,\ 0.5)$ | $0.09 + 0.04 + 0.25 = 0.38$ | $\mathbf{0.62}$ |

### Question 3 — Gini pondéré après chaque split
On pondère chaque enfant par sa **part d'observations** ($N_{\text{enfant}} / N_{\text{parent}}$, ici $10/20$) :
$$G_{\text{split}} = \frac{N_G}{N}\,G(t_G) + \frac{N_D}{N}\,G(t_D).$$
- Split A : $0.5 \times 0.56 + 0.5 \times 0.62 = \mathbf{0.59}$ ;
- Split B : $0.5 \times 0.50 + 0.5 \times 0.62 = \mathbf{0.56}$.

### Question 4 — Split choisi par CART
CART choisit le split qui **maximise la diminution d'impureté** $\Delta G = G(\text{parent}) - G_{\text{split}}$ :
- A : $0.655 - 0.59 = 0.065$ ;
- B : $0.655 - 0.56 = 0.095$.

→ **CART choisit le split B** (diminution la plus forte). Intuitivement, B isole complètement la classe 2 de
son enfant gauche (nœud gauche sans classe 2), alors que A laisse les trois classes mélangées des deux côtés.

In [7]:
def gini(effectifs):
    p = np.array(effectifs) / np.sum(effectifs)
    return 1 - np.sum(p ** 2)


parent = [8, 7, 5]
splits = {"A": ([6, 2, 2], [2, 5, 3]), "B": ([5, 5, 0], [3, 2, 5])}

print("Gini parent :", round(gini(parent), 4))
lignes = []
for nom, (gauche, droite) in splits.items():
    n_g, n_d = sum(gauche), sum(droite)
    pondere = (n_g * gini(gauche) + n_d * gini(droite)) / (n_g + n_d)
    lignes.append({"Split": nom, "Gini gauche": gini(gauche), "Gini droite": gini(droite),
                   "Gini pondere": pondere, "Diminution": gini(parent) - pondere})
pd.DataFrame(lignes).round(4)

Gini parent : 0.655


,Split,Gini gauche,Gini droite,Gini pondere,Diminution
0,A,0.56,0.62,0.59,0.065
1,B,0.50,0.62,0.56,0.095


---
## Problème 4 — Classification sensible aux coûts

Transaction avec $p(x) = P(Y = 1 \mid X = x) = 0.15$ (probabilité de fraude).
Coûts : **faux négatif** (fraude non détectée) $c_{FN} = 40$ ; **faux positif** (transaction légitime bloquée)
$c_{FP} = 6$ ; bonne décision : $0$.

**Principe.** Pour chaque décision possible, on calcule son **coût espéré** en pondérant chaque coût par la
probabilité que l'erreur correspondante se produise, puis on choisit la décision la **moins coûteuse**.

### Question 1 — Coût espéré de prédire 0 (légitime)
On se trompe seulement si la transaction est en fait une fraude (probabilité $p(x)$) :
$$\mathbb{E}[\text{coût} \mid \hat y = 0] = c_{FN}\,p(x) = 40 \times 0.15 = \mathbf{6}.$$

### Question 2 — Coût espéré de prédire 1 (fraude)
On se trompe seulement si la transaction est légitime (probabilité $1 - p(x)$) :
$$\mathbb{E}[\text{coût} \mid \hat y = 1] = c_{FP}\,(1 - p(x)) = 6 \times 0.85 = \mathbf{5.1}.$$

### Question 3 — Décision optimale
$5.1 < 6$ : il faut **prédire 1 (fraude)**, alors même que la fraude n'a que 15 % de chances d'être vraie.
Avec le seuil usuel de $1/2$, on aurait prédit « légitime » : le seuil $1/2$ n'est optimal que si les deux
erreurs coûtent la même chose.

### Question 4 — Seuil optimal
On prédit 1 dès que c'est moins coûteux (ou aussi coûteux) que prédire 0 :
$$c_{FP}\,(1 - p) \le c_{FN}\,p
\iff c_{FP} \le (c_{FP} + c_{FN})\,p
\iff \boxed{p \ge t^\star = \frac{c_{FP}}{c_{FP} + c_{FN}}}.$$
Ici $t^\star = \dfrac{6}{6 + 40} = \dfrac{3}{23} \approx \mathbf{0.130}$ : comme $0.15 \ge 0.130$, on retrouve la
décision « fraude ». Avec des coûts égaux, $t^\star = 1/2$ (règle de Bayes classique).

### Question 5 — Si $c_{FN}$ passe à 20
- Nouveau seuil : $t^\star = \dfrac{6}{6 + 20} = \dfrac{3}{13} \approx 0.231$ ;
- coûts espérés : prédire 0 → $20 \times 0.15 = 3$ ; prédire 1 → $6 \times 0.85 = 5.1$.

$0.15 < 0.231$ (de manière équivalente $3 < 5.1$) : la décision **bascule vers 0 (légitime)**. Rater une
fraude coûtant moins cher, il faut être plus sûr de la fraude avant de bloquer la transaction.

In [8]:
p, c_fp = 0.15, 6
for c_fn in (40, 20):
    cout_0, cout_1 = c_fn * p, c_fp * (1 - p)
    seuil = c_fp / (c_fp + c_fn)
    print(f"c_FN = {c_fn} : cout(0) = {cout_0:.2f} | cout(1) = {cout_1:.2f} | seuil t* = {seuil:.3f}"
          f" -> prediction {int(cout_1 <= cout_0)}")

c_FN = 40 : cout(0) = 6.00 | cout(1) = 5.10 | seuil t* = 0.130 -> prediction 1
c_FN = 20 : cout(0) = 3.00 | cout(1) = 5.10 | seuil t* = 0.231 -> prediction 0


In [9]:
grille = np.linspace(0, 1, 201)
fig = go.Figure()
fig.add_trace(go.Scatter(x=grille, y=40 * grille, name="Predire 0 : 40 p", line=dict(width=3, color=pick_color(0))))
fig.add_trace(go.Scatter(x=grille, y=20 * grille, name="Predire 0 : 20 p (c_FN = 20)",
                         line=dict(width=3, dash="dot", color=pick_color(0))))
fig.add_trace(go.Scatter(x=grille, y=6 * (1 - grille), name="Predire 1 : 6 (1 - p)",
                         line=dict(width=3, color=pick_color(1))))
fig.add_vline(x=0.15, line_dash="dash", line_color=HIGHLIGHT, annotation_text="p(x) = 0.15")
fig.update_layout(title="Probleme 4 : cout espere de chaque decision selon p(x)", xaxis_title="p(x) = P(fraude | x)",
                  yaxis_title="Cout espere", yaxis=dict(range=[0, 10]))
fig.show()

**Lecture.** La décision optimale est celle de la courbe **la plus basse**. Le seuil $t^\star$ est l'abscisse
du croisement des deux droites : $\approx 0.13$ avec $c_{FN} = 40$, $\approx 0.23$ avec $c_{FN} = 20$. Le point
$p = 0.15$ se trouve entre les deux, d'où le changement de décision.

---
## Problème 5 — Arbre de régression et erreur quadratique

| $X$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $Y$ | 2 | 4 | 3 | 7 | 8 | 6 |

Impureté d'un nœud : $I(t) = \dfrac{1}{N_t}\sum_{i \in t} (Y_i - \bar Y_t)^2$ (variance intra-nœud). Un arbre de
régression prédit, dans chaque feuille, la **moyenne** des $Y$ de la feuille (c'est la constante qui minimise
l'erreur quadratique).

### Question 1 — Prédiction avant découpage
$$\bar Y = \frac{2 + 4 + 3 + 7 + 8 + 6}{6} = \frac{30}{6} = \mathbf{5}.$$
Impureté de la racine : écarts $(-3, -1, -2, 2, 3, 1)$, carrés $(9, 1, 4, 4, 9, 1)$ de somme $28$, donc
$I(\text{racine}) = 28/6 \approx 4.667$.

### Question 2 — Split A : $X \le 3$
- gauche $\{2, 4, 3\}$ : $\bar Y_G = 9/3 = \mathbf{3}$ ;
- droite $\{7, 8, 6\}$ : $\bar Y_D = 21/3 = \mathbf{7}$.

### Question 3 — Split B : $X \le 4$
- gauche $\{2, 4, 3, 7\}$ : $\bar Y_G = 16/4 = \mathbf{4}$ ;
- droite $\{8, 6\}$ : $\bar Y_D = 14/2 = \mathbf{7}$.

### Question 4 — Impureté pondérée
**Split A**
- gauche : écarts $(-1, 1, 0)$ → $I_G = (1 + 1 + 0)/3 = 2/3$ ;
- droite : écarts $(0, 1, -1)$ → $I_D = (0 + 1 + 1)/3 = 2/3$ ;
- pondérée : $\frac{3}{6}\cdot\frac{2}{3} + \frac{3}{6}\cdot\frac{2}{3} = \mathbf{2/3 \approx 0.667}$.

**Split B**
- gauche : écarts $(-2, 0, -1, 3)$ → $I_G = (4 + 0 + 1 + 9)/4 = 14/4 = 3.5$ ;
- droite : écarts $(1, -1)$ → $I_D = (1 + 1)/2 = 1$ ;
- pondérée : $\frac{4}{6}\times 3.5 + \frac{2}{6}\times 1 = \frac{14 + 2}{6} = \mathbf{8/3 \approx 2.667}$.

### Question 5 — Meilleur split
Réduction d'impureté :
- A : $4.667 - 0.667 = \mathbf{4}$ ;
- B : $4.667 - 2.667 = \mathbf{2}$.

→ **Le split A** réduit le plus l'erreur quadratique. En effet, l'observation $(4, 7)$ ressemble aux grandes
valeurs ($7, 8, 6$) : B la met avec les petites valeurs, ce qui crée beaucoup de dispersion à gauche.

En termes de somme des carrés : on passe de $28$ à $2 + 2 = 4$ avec A, contre $14 + 2 = 16$ avec B.

### Question 6 — Prédiction pour $X = 3.5$
Avec le split A, $3.5 > 3$ → feuille **droite** → $\hat Y = \mathbf{7}$.

In [10]:
X = np.array([1, 2, 3, 4, 5, 6])
Y = np.array([2, 4, 3, 7, 8, 6])


def impurete(y):
    return np.mean((y - y.mean()) ** 2)


print("Prediction sans split :", Y.mean(), "| impurete racine :", round(impurete(Y), 3))
for nom, s in [("A", 3), ("B", 4)]:
    g, d = Y[X <= s], Y[X > s]
    pondere = (len(g) * impurete(g) + len(d) * impurete(d)) / len(Y)
    print(f"Split {nom} (X <= {s}) : moyennes {g.mean():.2f} / {d.mean():.2f} | "
          f"impurete ponderee {pondere:.3f} | reduction {impurete(Y) - pondere:.3f}")

Prediction sans split : 5.0 | impurete racine : 4.667
Split A (X <= 3) : moyennes 3.00 / 7.00 | impurete ponderee 0.667 | reduction 4.000
Split B (X <= 4) : moyennes 4.00 / 7.00 | impurete ponderee 2.667 | reduction 2.000


In [11]:
fig = go.Figure(go.Scatter(x=X, y=Y, mode="markers", marker=dict(size=13, color=pick_color(0)), name="Observations"))
for i, (nom, s) in enumerate([("A : X <= 3", 3), ("B : X <= 4", 4)]):
    g, d = Y[X <= s].mean(), Y[X > s].mean()
    fig.add_trace(go.Scatter(x=[0.5, s + 0.5, s + 0.5, 6.5], y=[g, g, d, d], mode="lines", line_shape="hv",
                             name=f"Split {nom}", line=dict(width=3, color=pick_color(i + 1))))
fig.add_hline(y=Y.mean(), line_dash="dash", line_color=MUTED, annotation_text="moyenne globale = 5")
fig.update_layout(title="Probleme 5 : predictions en escalier des deux splits", xaxis_title="X", yaxis_title="Y")
fig.show()

---
## Problème 6 — Matrice de confusion et mesures de performance

300 observations, dont **60 positives** (donc 240 négatives). Le classifieur détecte correctement **45
positifs** et classe à tort **30 négatifs** comme positifs.

### Question 1 — Matrice de confusion
- $VP = 45$ ; $FN = 60 - 45 = 15$ (positifs manqués) ;
- $FP = 30$ ; $VN = 240 - 30 = 210$.

| | Prédit positif | Prédit négatif | Total |
|---|---|---|---|
| **Réel positif** | VP = 45 | FN = 15 | 60 |
| **Réel négatif** | FP = 30 | VN = 210 | 240 |
| **Total** | 75 | 225 | 300 |

### Question 2 — Mesures
| Mesure | Formule | Calcul | Valeur |
|---|---|---|---|
| Accuracy | $\frac{VP + VN}{n}$ | $\frac{45 + 210}{300}$ | **0.85** |
| Précision | $\frac{VP}{VP + FP}$ | $\frac{45}{75}$ | **0.60** |
| Rappel (TPR) | $\frac{VP}{VP + FN}$ | $\frac{45}{60}$ | **0.75** |
| FPR | $\frac{FP}{FP + VN}$ | $\frac{30}{240}$ | **0.125** |

Interprétation : 75 % des positifs sont détectés, mais seulement 60 % des alertes sont justifiées.

### Question 3 — Score $F_1$
$$F_1 = \frac{2 \times 0.60 \times 0.75}{0.60 + 0.75} = \frac{0.90}{1.35} = \frac{2}{3} \approx \mathbf{0.667}.$$
(Équivalent : $F_1 = \frac{2\,VP}{2\,VP + FP + FN} = \frac{90}{135}$.) Moyenne **harmonique** : elle est tirée
vers la plus faible des deux valeurs (ici la précision).

### Question 4 — Classifieur « toujours négatif »
Il prédit négatif pour les 300 observations : $VP = 0$, $FN = 60$, $FP = 0$, $VN = 240$.
- **Accuracy** $= 240/300 = \mathbf{0.80}$ ;
- **Rappel** $= 0/60 = \mathbf{0}$ (précision non définie : aucune prédiction positive ; $F_1 = 0$).

### Question 5 — Pourquoi l'accuracy seule est trompeuse
Le classifieur trivial obtient $0.80$ d'accuracy **sans rien apprendre** et sans détecter un seul positif,
soit seulement 5 points de moins que notre classifieur ($0.85$) qui, lui, en détecte 75 %. Quand une classe est
**majoritaire** (ici 80 % de négatifs), l'accuracy est dominée par cette classe : elle récompense le fait de
prédire la classe majoritaire et **ignore le coût de rater les positifs**, qui sont souvent la classe
d'intérêt (fraude, maladie…). Il faut donc compléter par le rappel, la précision, le $F_1$, la courbe ROC /
l'AUC, et comparer à la **référence triviale** (taux de la classe majoritaire).

In [12]:
vp, fn, fp, vn = 45, 15, 30, 210
print(pd.DataFrame([[vp, fn], [fp, vn]], index=["Reel +", "Reel -"], columns=["Predit +", "Predit -"]))

precision, rappel = vp / (vp + fp), vp / (vp + fn)
print(f"\nAccuracy = {(vp + vn) / 300:.3f} | Precision = {precision:.3f} | Rappel = {rappel:.3f} | "
      f"FPR = {fp / (fp + vn):.3f} | F1 = {2 * precision * rappel / (precision + rappel):.3f}")
print(f"Toujours negatif : accuracy = {240 / 300:.3f} | rappel = {0 / 60:.3f}")

        Predit +  Predit -
Reel +        45        15
Reel -        30       210

Accuracy = 0.850 | Precision = 0.600 | Rappel = 0.750 | FPR = 0.125 | F1 = 0.667
Toujours negatif : accuracy = 0.800 | rappel = 0.000


---
## Problème 7 — Courbes ROC et AUC

| Observation | A | B | C | D | E | F |
|---|---|---|---|---|---|---|
| Vraie classe | 1 | 0 | 1 | 0 | 1 | 0 |
| Score | 0.90 | 0.80 | 0.70 | 0.60 | 0.40 | 0.20 |

3 positifs (A, C, E) et 3 négatifs (B, D, F). On prédit 1 si $s(x) \ge t$ ;
$\text{TPR} = VP/3$ et $\text{FPR} = FP/3$.

### Question 1 — TPR et FPR pour chaque seuil
| Seuil $t$ | Prédits positifs | VP | FP | TPR | FPR |
|---|---|---|---|---|---|
| 0.85 | A | 1 | 0 | $1/3 \approx 0.333$ | $0$ |
| 0.65 | A, B, C | 2 | 1 | $2/3 \approx 0.667$ | $1/3 \approx 0.333$ |
| 0.45 | A, B, C, D | 2 | 2 | $2/3 \approx 0.667$ | $2/3 \approx 0.667$ |
| 0.30 | A, B, C, D, E | 3 | 2 | $1$ | $2/3 \approx 0.667$ |

### Question 2 — Points ROC
Avec les extrémités : $(0, 0)$ (seuil $> 0.90$, rien n'est positif) et $(1, 1)$ (seuil $\le 0.20$, tout est
positif), les points $(\text{FPR}, \text{TPR})$ sont :
$$(0, 0),\ (0, \tfrac13),\ (\tfrac13, \tfrac23),\ (\tfrac23, \tfrac23),\ (\tfrac23, 1),\ (1, 1).$$
Le graphique ci-dessous trace aussi la **courbe ROC complète** (en escalier, tous les seuils).

In [13]:
y_vrai = np.array([1, 0, 1, 0, 1, 0])
score = np.array([0.90, 0.80, 0.70, 0.60, 0.40, 0.20])


def tpr_fpr(t):
    pred = score >= t
    return pred[y_vrai == 1].mean(), pred[y_vrai == 0].mean()


for t in (0.85, 0.65, 0.45, 0.30):
    tpr, fpr = tpr_fpr(t)
    print(f"t = {t:.2f} : TPR = {tpr:.3f} | FPR = {fpr:.3f}")

t = 0.85 : TPR = 0.333 | FPR = 0.000
t = 0.65 : TPR = 0.667 | FPR = 0.333
t = 0.45 : TPR = 0.667 | FPR = 0.667
t = 0.30 : TPR = 1.000 | FPR = 0.667


In [14]:
# Courbe complete : on abaisse le seuil score par score (de +inf a la plus petite valeur)
seuils = np.r_[np.inf, np.sort(score)[::-1]]
courbe = np.array([tpr_fpr(t) for t in seuils])
points = np.array([(0, 0)] + [tpr_fpr(t) for t in (0.85, 0.65, 0.45, 0.30)] + [(1, 1)])

fig = go.Figure()
fig.add_trace(go.Scatter(x=courbe[:, 1], y=courbe[:, 0], mode="lines", line_shape="hv", name="Courbe ROC",
                         line=dict(width=3, color=pick_color(0)), fill="tozeroy"))
fig.add_trace(go.Scatter(x=points[:, 1], y=points[:, 0], mode="markers", name="Points demandes",
                         marker=dict(size=13, color=HIGHLIGHT)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatoire", line=dict(color=MUTED, dash="dash")))
fig.update_layout(title="Probleme 7 : courbe ROC (aire coloree = AUC = 2/3)", xaxis_title="FPR", yaxis_title="TPR",
                  yaxis=dict(scaleanchor="x"), height=550)
fig.show()

### Question 3 — AUC empirique par comparaison des paires
On compare chaque positif à chaque négatif ($3 \times 3 = 9$ paires). Une paire est **concordante** si le
positif a le score le plus élevé (une égalité compterait $1/2$ ; il n'y en a pas ici).

| Positif \ Négatif | B (0.80) | D (0.60) | F (0.20) | Concordantes |
|---|---|---|---|---|
| A (0.90) | ✓ | ✓ | ✓ | 3 |
| C (0.70) | ✗ | ✓ | ✓ | 2 |
| E (0.40) | ✗ | ✗ | ✓ | 1 |

$$\text{AUC} = \frac{3 + 2 + 1}{9} = \frac{6}{9} = \frac{2}{3} \approx \mathbf{0.667}.$$
Vérification géométrique : l'aire sous l'escalier vaut
$\frac13 \times \frac13 + \frac13 \times \frac23 + \frac13 \times 1 = \frac{1 + 2 + 3}{9} = \frac23$.

### Question 4 — Interprétation probabiliste
$\text{AUC} = P\big(s(X^+) > s(X^-)\big)$ : c'est la probabilité qu'un **positif tiré au hasard** reçoive un
score **plus élevé** qu'un **négatif tiré au hasard**. Ici, dans **2 cas sur 3**, le classifieur ordonne
correctement une paire (positif, négatif). Repères : $0.5$ = classement aléatoire, $1$ = séparation parfaite.

### Question 5 — Scores remplacés par $2\,s(x) + 1$
La transformation $s \mapsto 2s + 1$ est **strictement croissante** : elle conserve l'**ordre** de tous les
scores. Or la courbe ROC ne dépend que de cet ordre : le seuil $t$ sur $s$ correspond exactement au seuil
$2t + 1$ sur les nouveaux scores, avec les mêmes prédictions. **La courbe ROC et l'AUC sont donc
inchangées.** (Seules les valeurs des seuils changent ; les nouveaux scores, entre 1.4 et 2.8, ne sont
d'ailleurs plus des probabilités, ce qui n'a aucune importance pour la ROC.)

In [15]:
pos, neg = score[y_vrai == 1], score[y_vrai == 0]
paires = (pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :])
print("AUC par paires :", round(paires.mean(), 4))
print("AUC sklearn    :", round(roc_auc_score(y_vrai, score), 4))
print("AUC avec 2s+1  :", round(roc_auc_score(y_vrai, 2 * score + 1), 4))

AUC par paires : 0.6667
AUC sklearn    : 0.6667
AUC avec 2s+1  : 0.6667


---
## Problème 8 — Régression logistique et frontières de décision

$$p(x_1, x_2) = \frac{1}{1 + \exp[-(x_1 + 2x_2 - 3)]} = \sigma(z), \qquad z = x_1 + 2x_2 - 3,$$
où $\sigma(z) = 1/(1 + e^{-z})$ est la fonction **sigmoïde**, strictement croissante, avec $\sigma(0) = 0.5$.
On prédit 1 si $p(x_1, x_2) \ge t$.

**Idée clé.** Comme $\sigma$ est croissante, $p \ge t \iff z \ge \operatorname{logit}(t) = \ln\frac{t}{1 - t}$.
Le seuil sur la probabilité devient un seuil sur la **fonction linéaire** $z$ : la frontière est une **droite**.

### Question 1 — Frontière pour $t = 0.5$
$\operatorname{logit}(0.5) = \ln 1 = 0$, donc
$$p \ge 0.5 \iff x_1 + 2x_2 - 3 \ge 0 \iff \boxed{x_1 + 2x_2 \ge 3}.$$
La frontière est la droite $x_1 + 2x_2 = 3$ (soit $x_2 = (3 - x_1)/2$) ; la classe 1 est du côté où
$x_1 + 2x_2$ est grand.

### Question 2 — Probabilités prédites
- $A = (1, 1)$ : $z = 1 + 2 - 3 = 0$, donc $p(A) = \sigma(0) = \mathbf{0.5}$ — A est **exactement sur la frontière**.
  Avec la règle $p \ge 0.5$, A est classé **1** (cas limite).
- $B = (3, 1)$ : $z = 3 + 2 - 3 = 2$, donc $p(B) = \dfrac{1}{1 + e^{-2}} \approx \mathbf{0.881}$ → classe **1**.

### Question 3 — Seuil donnant la frontière $x_1 + 2x_2 = 4$
$x_1 + 2x_2 = 4 \iff z = 1$. Il faut donc $\operatorname{logit}(t) = 1$, c'est-à-dire
$$t = \sigma(1) = \frac{1}{1 + e^{-1}} \approx \mathbf{0.731}.$$

### Question 4 — Prédictions avec $t \approx 0.731$
- A : $p = 0.5 < 0.731$ → classe **0** (A est maintenant du mauvais côté : $1 + 2 = 3 < 4$) ;
- B : $p \approx 0.881 \ge 0.731$ → classe **1** ($3 + 2 = 5 \ge 4$).

### Question 5 — Interprétation géométrique
Pour tout seuil $t$, la frontière est la droite
$$x_1 + 2x_2 = 3 + \ln\frac{t}{1 - t}.$$
- Toutes ces droites sont **parallèles** : elles ont le même vecteur normal $(1, 2)$ (les coefficients du
  modèle). Changer $t$ ne fait pas **tourner** la frontière, il la **translate**.
- **Augmenter** $t$ décale la droite dans la direction de $(1, 2)$, vers la zone des fortes probabilités : la
  région prédite 1 **rétrécit** (moins de faux positifs, plus de faux négatifs). Diminuer $t$ fait l'inverse.
- Ici, passer de $t = 0.5$ à $t \approx 0.731$ translate la droite d'une distance $1/\|(1, 2)\| = 1/\sqrt5 \approx 0.447$.

In [16]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


points = {"A": (1, 1), "B": (3, 1)}
t_new = sigmoide(1)
print(f"Nouveau seuil t = sigmoide(1) = {t_new:.4f}")
for nom, (x1, x2) in points.items():
    p = sigmoide(x1 + 2 * x2 - 3)
    print(f"{nom} : p = {p:.4f} | classe (t = 0.5) = {int(p >= 0.5)} | classe (t = {t_new:.3f}) = {int(p >= t_new)}")

Nouveau seuil t = sigmoide(1) = 0.7311
A : p = 0.5000 | classe (t = 0.5) = 1 | classe (t = 0.731) = 0
B : p = 0.8808 | classe (t = 0.5) = 1 | classe (t = 0.731) = 1


In [17]:
x1 = np.linspace(-1, 6, 50)
fig = go.Figure()
for i, (t, c) in enumerate([(0.5, 3), (round(t_new, 3), 4)]):
    fig.add_trace(go.Scatter(x=x1, y=(c - x1) / 2, mode="lines", name=f"t = {t} : x1 + 2 x2 = {c}",
                             line=dict(width=3, color=pick_color(i))))
fig.add_trace(go.Scatter(x=[1, 3], y=[1, 1], mode="markers+text", text=["A", "B"], textposition="top center",
                         marker=dict(size=14, color=HIGHLIGHT), name="Points A et B"))
fig.update_layout(title="Probleme 8 : frontieres paralleles selon le seuil (classe 1 au-dessus)",
                  xaxis_title="x1", yaxis_title="x2", yaxis=dict(scaleanchor="x", range=[-1, 3.5]), height=500)
fig.show()

---
## Problème 9 — Perte logistique et erreur de classification

| Obs. | Vraie classe | Modèle A | Modèle B |
|---|---|---|---|
| 1 | 1 | 0.55 | 0.90 |
| 2 | 0 | 0.45 | 0.30 |
| 3 | 1 | 0.60 | 0.80 |
| 4 | 0 | 0.40 | 0.70 |

Seuil $0.5$. Perte logistique : $L = -\frac1n \sum_i \big[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\big]$, c'est-à-dire
la moyenne de $-\ln(\text{probabilité attribuée à la vraie classe})$.

### Question 1 — Erreur de classification
- **Modèle A** : prédictions $(1, 0, 1, 0)$ → toutes correctes → erreur $= 0/4 = \mathbf{0}$.
- **Modèle B** : prédictions $(1, 0, 1, 1)$ → l'observation 4 (classe 0, $p = 0.70$) est mal classée →
  erreur $= 1/4 = \mathbf{0.25}$.

### Question 2 — Perte logistique moyenne
Probabilité attribuée à la **vraie classe** (c'est $p_i$ si $y_i = 1$, $1 - p_i$ si $y_i = 0$) :

| Obs. | Modèle A | $-\ln$ | Modèle B | $-\ln$ |
|---|---|---|---|---|
| 1 ($y=1$) | 0.55 | 0.598 | 0.90 | 0.105 |
| 2 ($y=0$) | 0.55 | 0.598 | 0.70 | 0.357 |
| 3 ($y=1$) | 0.60 | 0.511 | 0.80 | 0.223 |
| 4 ($y=0$) | 0.60 | 0.511 | 0.30 | 1.204 |
| **Somme** | | 2.217 | | 1.889 |

- $L_A = 2.217 / 4 \approx \mathbf{0.554}$ ;
- $L_B = 1.889 / 4 \approx \mathbf{0.472}$.

### Question 3 — Comparaison
- Plus faible **erreur de classification** : **modèle A** ($0$ contre $0.25$) ;
- plus faible **perte logistique** : **modèle B** ($0.472$ contre $0.554$).

### Question 4 — Comment est-ce possible ?
Les deux critères ne mesurent pas la même chose :
- l'**erreur de classification** ne regarde que le **côté du seuil** où tombe $p_i$ : $0.55$ et $0.99$ comptent
  pareil s'ils sont du bon côté ;
- la **perte logistique** mesure la **confiance** accordée à la vraie classe : elle pénalise les prédictions
  hésitantes et, très fortement, les erreurs commises avec assurance ($-\ln p \to +\infty$ quand $p \to 0$).

Le modèle A a toujours raison mais **de justesse** (probabilités $0.55$–$0.60$, proches de $0.5$) : il paie
environ $0.55$ sur **chaque** observation. Le modèle B est **très confiant et juste** sur trois observations
(pertes faibles : $0.11$, $0.36$, $0.22$) et se trompe une fois (perte $1.20$) ; au total, sa perte moyenne est
plus faible. La log-loss évalue la **qualité des probabilités**, l'erreur évalue la **qualité des décisions**
à un seuil donné.

### Question 5 — Le seuil influence-t-il la perte logistique ?
**Non.** La perte logistique est calculée directement à partir des **probabilités** $p_i$ et des vraies classes
$y_i$ ; le seuil n'intervient nulle part dans la formule. Changer le seuil modifie les **prédictions** (donc
l'erreur de classification, la précision, le rappel…), mais pas $L$. C'est pourquoi on entraîne la régression
logistique en minimisant la log-loss, puis on choisit le seuil séparément (selon les coûts, par exemple).

In [18]:
y = np.array([1, 0, 1, 0])
probas = {"A": np.array([0.55, 0.45, 0.60, 0.40]), "B": np.array([0.90, 0.30, 0.80, 0.70])}

for nom, p in probas.items():
    erreur = np.mean((p >= 0.5) != y)
    perte = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    print(f"Modele {nom} : erreur = {erreur:.2f} | log-loss = {perte:.4f} (sklearn : {log_loss(y, p):.4f})")

Modele A : erreur = 0.00 | log-loss = 0.5543 (sklearn : 0.5543)
Modele B : erreur = 0.25 | log-loss = 0.4723 (sklearn : 0.4723)


---
## Problème 10 — Validation croisée et sélection de modèle

Validation croisée à 4 plis pour $k \in \{1, 5, 15\}$, chaque pli de validation contenant **50** observations.

| Modèle | Pli 1 | Pli 2 | Pli 3 | Pli 4 |
|---|---|---|---|---|
| $k = 1$ | 4 | 8 | 5 | 7 |
| $k = 5$ | 5 | 4 | 6 | 3 |
| $k = 15$ | 7 | 5 | 8 | 6 |

### Question 1 — Taux d'erreur moyen de validation
Chaque modèle est évalué sur $4 \times 50 = 200$ observations au total :

| $k$ | Total d'erreurs | Taux moyen | Taux par pli |
|---|---|---|---|
| 1 | $4 + 8 + 5 + 7 = 24$ | $24/200 = \mathbf{0.120}$ | 0.08, 0.16, 0.10, 0.14 |
| 5 | $5 + 4 + 6 + 3 = 18$ | $18/200 = \mathbf{0.090}$ | 0.10, 0.08, 0.12, 0.06 |
| 15 | $7 + 5 + 8 + 6 = 26$ | $26/200 = \mathbf{0.130}$ | 0.14, 0.10, 0.16, 0.12 |

(Les plis ayant la même taille, la moyenne des 4 taux est égale au total d'erreurs divisé par 200.)

### Question 2 — Valeur retenue
On retient le $k$ de **plus faible erreur de validation** : **$k = 5$** ($9\,\%$).
C'est l'illustration du compromis biais–variance : $k = 1$ colle trop aux données (variance élevée,
surapprentissage), $k = 15$ lisse trop (biais élevé, sous-apprentissage).

### Question 3 — Taille de l'apprentissage dans chaque pli
- $250$ observations dont $50$ réservées au **test** → $200$ pour la validation croisée ;
- 4 plis de $200/4 = 50$ observations ;
- à chaque itération : 1 pli en validation, les 3 autres en apprentissage → $3 \times 50 = \mathbf{150}$
  observations d'apprentissage.

### Question 4 — Apprentissage du modèle final
Une fois $k = 5$ choisi, on ré-entraîne le modèle sur **toutes les données hors test**, soit les
**200 observations** (apprentissage + validation). Plus de données donne un meilleur modèle, et
l'hyperparamètre est déjà fixé : on n'a plus besoin de mettre des données de côté pour la sélection.
Le jeu de test (50 observations) reste **intact** jusqu'à l'évaluation finale.

### Question 5 — Erreur de test et pourquoi ne pas s'en servir pour choisir $k$
$$\text{Erreur de test} = \frac{7}{50} = \mathbf{0.14}.$$
Cette valeur est une estimation **sans biais** de l'erreur de généralisation **uniquement parce que** le jeu de
test n'a joué **aucun rôle** dans les choix. Si on l'utilise pour changer $k$ (« 14 %, c'est trop, essayons
$k = 15$ »), le test devient un jeu de **validation** : on choisirait le $k$ qui marche le mieux **sur ces 50
observations précises**, en partie grâce au hasard, et l'erreur affichée serait **optimiste** (biaisée vers
le bas). On n'aurait alors plus aucune estimation honnête des performances sur de nouvelles données. La
sélection se fait par validation croisée ; le test sert **une seule fois**, à la fin.

On note d'ailleurs que $0.14 > 0.09$ : l'erreur de validation du $k$ retenu est elle-même légèrement
optimiste (on a pris le minimum de trois estimations bruitées), et 50 observations de test donnent une
estimation assez variable.

In [19]:
erreurs = pd.DataFrame({"Pli 1": [4, 5, 7], "Pli 2": [8, 4, 5], "Pli 3": [5, 6, 8], "Pli 4": [7, 3, 6]},
                       index=pd.Index([1, 5, 15], name="k"))
taux = erreurs / 50
resume = pd.DataFrame({"Erreurs totales": erreurs.sum(axis=1), "Taux moyen": taux.mean(axis=1),
                       "Ecart-type entre plis": taux.std(axis=1)}).round(3)
print(resume)
print("\nk retenu :", resume["Taux moyen"].idxmin())
print("Apprentissage par pli :", (250 - 50) * 3 // 4, "| modele final :", 250 - 50, "| erreur test :", 7 / 50)

    Erreurs totales  Taux moyen  Ecart-type entre plis
k                                                     
1                24        0.12                  0.037
5                18        0.09                  0.026
15               26        0.13                  0.026

k retenu : 5
Apprentissage par pli : 150 | modele final : 200 | erreur test : 0.14


---
## Récapitulatif des résultats

| Problème | Résultats principaux |
|---|---|
| 1 — $k$-NN | $d = (5;\ 1.414;\ 1.414;\ 3.606;\ 5)$ ; 1-NN → 1, 3-NN → 1 ; après $X_1 \times 10$ : prédictions identiques mais distances dominées par $X_1$ → **standardiser** |
| 2 — Bayes naïf | $0.144$ et $0.036$ ; $P(Y = 1 \mid x) = 12/19 \approx 0.632$ → classe **1** |
| 3 — Gini | parent $0.655$ ; A : $0.56 / 0.62$ → $0.59$ ; B : $0.50 / 0.62$ → $0.56$ → split **B** |
| 4 — Coûts | coût(0) $= 6$, coût(1) $= 5.1$ → **fraude** ; $t^\star = c_{FP}/(c_{FP} + c_{FN}) \approx 0.130$ ; si $c_{FN} = 20$ : $t^\star \approx 0.231$ → **légitime** |
| 5 — Arbre de régression | racine $5$ ; A : $3 / 7$, impureté $0.667$ ; B : $4 / 7$, impureté $2.667$ → split **A** ; $\hat Y(3.5) = 7$ |
| 6 — Confusion | VP 45, FN 15, FP 30, VN 210 ; acc $0.85$, préc $0.60$, rappel $0.75$, FPR $0.125$, $F_1 = 0.667$ ; trivial : acc $0.80$, rappel $0$ |
| 7 — ROC | $(0, \frac13), (\frac13, \frac23), (\frac23, \frac23), (\frac23, 1)$ ; AUC $= 6/9 = 2/3$ ; invariante par $2s + 1$ |
| 8 — Logistique | frontière $x_1 + 2x_2 = 3$ ; $p(A) = 0.5$, $p(B) \approx 0.881$ ; $t = \sigma(1) \approx 0.731$ → A : 0, B : 1 ; droites parallèles |
| 9 — Log-loss | erreur : A $0$, B $0.25$ ; log-loss : A $0.554$, B $0.472$ ; indépendante du seuil |
| 10 — Validation croisée | taux $0.12 / 0.09 / 0.13$ → **$k = 5$** ; 150 par pli ; modèle final sur 200 ; test $= 0.14$, à ne pas réutiliser |